# Customer churn with an ANN in Keras
Binary classification: will a bank customer leave? A first network (11-3-1, sigmoid), then a bigger one (11-11-11-1, ReLU) with a validation split and training curves.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's log messages
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"  # take GPU memory only as needed
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# same random numbers on every run: weights, shuffling, validation split
keras.utils.set_random_seed(1)
tf.config.experimental.enable_op_determinism()
print(tf.__version__, keras.__version__)

## The data
10,000 customers of a bank, 14 columns. `Exited` = 1 means the customer left.

In [ ]:
df = pd.read_csv("data/Churn_Modelling.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# duplicated rows, and how many customers left (1) or stayed (0)
print("duplicates:", df.duplicated().sum())
print(df["Exited"].value_counts())
print(df["Geography"].value_counts())
print(df["Gender"].value_counts())

## Preparing the inputs

In [ ]:
# row number, customer ID and surname say nothing about churn
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])
# one-hot encode the two text columns, keeping n - 1 columns each
# dtype=int gives 0/1 instead of True/False
df = pd.get_dummies(df, columns=["Geography", "Gender"], drop_first=True, dtype=int)
df.head()

In [ ]:
X = df.drop(columns="Exited")
y = df["Exited"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
print(X_train.shape, X_test.shape)
print(list(X.columns))

In [ ]:
# standardize: learn mean and std on the training set only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled[:2].round(2)

## First network: 11 inputs, 3 hidden nodes, 1 output

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(11,)),                 # 11 input columns
    layers.Dense(3, activation="sigmoid"),    # hidden layer
    layers.Dense(1, activation="sigmoid"),    # output: probability of leaving
])
model.summary()

In [ ]:
# loss for two classes, and the optimizer that updates the weights
model.compile(loss="binary_crossentropy", optimizer="adam")
# 10 passes over the 8,000 training rows
history1 = model.fit(X_train_scaled, y_train, epochs=10, verbose=2)

In [ ]:
# the 40 numbers training found
W1, b1 = model.layers[0].get_weights()
W2, b2 = model.layers[1].get_weights()
print(W1.shape, b1.shape, W2.shape, b2.shape)
print(W2.ravel().round(3), b2.round(3))

In [ ]:
# the output is a probability, not a class
y_log = model.predict(X_test_scaled, verbose=0)
print(y_log[:5].ravel().round(3))
print("lowest and highest:", y_log.min().round(3), y_log.max().round(3))
# threshold 0.5: above means "will leave" (1)
y_pred = np.where(y_log > 0.5, 1, 0)
acc1 = accuracy_score(y_test, y_pred)
cm1 = confusion_matrix(y_test, y_pred)
print("accuracy:", acc1)
print("customers predicted to leave:", y_pred.sum())

In [ ]:
# for comparison: always predicting "stays" (0)
print("always 0:", accuracy_score(y_test, np.zeros_like(y_test)))

## Second network: ReLU, 11 nodes, two hidden layers, 100 epochs, validation split

In [ ]:
model2 = keras.Sequential([
    keras.Input(shape=(11,)),
    layers.Dense(11, activation="relu"),
    layers.Dense(11, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model2.summary()

In [ ]:
# metrics=["accuracy"] adds accuracy to the per-epoch report
model2.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
# validation_split=0.2: the last 20% of the training rows are held back each epoch
history = model2.fit(X_train_scaled, y_train, epochs=100, validation_split=0.2, verbose=2)

In [ ]:
y_log2 = model2.predict(X_test_scaled, verbose=0)
y_pred2 = np.where(y_log2 > 0.5, 1, 0)
acc2 = accuracy_score(y_test, y_pred2)
cm = confusion_matrix(y_test, y_pred2)
print("accuracy:", acc2)
print(cm)

In [ ]:
# the History object: one list per quantity, one value per epoch
print(history.history.keys())
h = history.history
print({k: round(v[-1], 4) for k, v in h.items()})

## Training curves

In [ ]:
epochs = list(range(1, len(h["loss"]) + 1))
fig = go.Figure()
fig.add_scatter(x=epochs, y=h["loss"], name="training loss")
fig.add_scatter(x=epochs, y=h["val_loss"], name="validation loss")
fig.update_layout(template="simple_white", xaxis_title="epoch", yaxis_title="binary cross-entropy")
fig.show()

In [ ]:
fig = go.Figure()
fig.add_scatter(x=epochs, y=h["accuracy"], name="training accuracy")
fig.add_scatter(x=epochs, y=h["val_accuracy"], name="validation accuracy")
fig.update_layout(template="simple_white", xaxis_title="epoch", yaxis_title="accuracy")
fig.show()

In [ ]:
# save the numbers the Note's figures are drawn from (images/figs.py)
results = {"history1": history1.history, "history": h, "cm": cm.tolist(), "cm1": cm1.tolist(),
           "acc1": acc1, "acc2": acc2}
Path("images/results.json").write_text(json.dumps(results))

## The first network from other random starts
Same 11-3-1 network and 10 epochs, five different seeds; then seed 1 with 100 epochs.

In [ ]:
for seed, epochs in [(0, 10), (1, 10), (2, 10), (3, 10), (4, 10), (1, 100)]:
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([keras.Input(shape=(11,)), layers.Dense(3, activation="sigmoid"),
                          layers.Dense(1, activation="sigmoid")])
    m.compile(loss="binary_crossentropy", optimizer="adam")
    m.fit(X_train_scaled, y_train, epochs=epochs, verbose=0)
    pred = (m.predict(X_test_scaled, verbose=0).ravel() > 0.5).astype(int)
    print(f"seed {seed}, {epochs} epochs: accuracy {np.mean(pred == y_test):.4f}, flagged as leaving {pred.sum()}")
keras.utils.set_random_seed(1)   # back to the seed used above
